In [16]:
%load_ext autoreload
%autoreload 2

import os
import glob
import yaml
import warnings; warnings.filterwarnings('ignore')
from dotenv import load_dotenv
import sys
sys.path.append('..') # src 폴더 인식을 위한 경로 추가
from src import build_parent_retriever, run_corrective_agent, extract_target_company
from langchain_classic.retrievers import (
    EnsembleRetriever,
    BM25Retriever,
)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [17]:
load_dotenv(dotenv_path="../.env") 
with open("../config/config.yaml", "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

gemini_api_key = os.getenv(config['api_keys']['google_gemini_env_name'])
if gemini_api_key:
    os.environ["GOOGLE_API_KEY"] = gemini_api_key
    print("✅ Gemini API 키 로드 완료")
else:
    print("❌ API 키 오류")

✅ Gemini API 키 로드 완료


In [18]:
data_dir = "../data/raw/"
pdf_files = glob.glob(os.path.join(data_dir, "*.pdf"))

print(f"📂 {len(pdf_files)}개의 PDF 문서에 대한 고급 검색 엔진을 구축합니다...")

# src/document_processor.py 에 정의된 함수 단 한 줄로 복잡한 DB 구축 끝!
retriever, all_docs = build_parent_retriever(pdf_files, chunk_size=200, force_rebuild=False, k=10)

📂 10개의 PDF 문서에 대한 고급 검색 엔진을 구축합니다...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 33167.75it/s]
RobertaModel LOAD REPORT from: jhgan/ko-sroberta-multitask
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


⚡ 하드디스크에 저장된 DB와 문서를 불러옵니다!
🔍 BM25 키워드 검색기 복원 중...
✅ 하이브리드 검색기(캐시) 로드 완료!


In [21]:
# ==========================================
# 🧠 0단계: Self-Query Router (지능형 사전 필터링)
# ==========================================
print("\n" + "="*50)
print("🧐 SOTA 하이브리드 에이전트 테스트 (Self-Query + Parent + BM25)")

query = "LG에너지솔루션의 연결 제무재표 기준 2025년도의 당기순이익은 얼마인가요?"
print(f"질문: {query}\n")

# 단 한 줄로 깔끔하게 타겟 기업 자동 추출!
extracted_company = extract_target_company(query)

if extracted_company != "ALL":
    print(f"🎯 자동 필터링 조건 생성 완료: {{'company': '{extracted_company}'}}")
    print(f"🛠️ '{extracted_company}' 전용 하이브리드 검색기로 실시간 재조립합니다...")
    
    # [사전 필터링] 사냥개가 처음부터 해당 기업 문서만 찾도록 강제 개조 (Pre-Retrieval)
    company_docs = [doc for doc in all_docs if extracted_company in doc.metadata.get('source', '')]
    
    if not company_docs:
        print(f"⚠️ 경고: '{extracted_company}'와 매칭되는 문서가 없으므로 전체 검색을 진행합니다.")
        active_retriever = retriever
    else:
        bm25_new = BM25Retriever.from_documents(company_docs)
        bm25_new.k = 10
        
        parent_old = retriever.retrievers[1] 
        parent_old.search_kwargs["filter"] = {"source": {"$contains": extracted_company}}
        
        active_retriever = EnsembleRetriever(retrievers=[bm25_new, parent_old], weights=[0.5, 0.5])
else:
    print("🌐 전체 기업 대상으로 검색을 진행합니다.")
    active_retriever = retriever

# ==========================================
# 🔍 1단계: 하이브리드 검색 실행 (순도 100% 정답 문서만 등장)
# ==========================================
print("\n🔍 [Path A] 사전 필터링이 적용된 상태로 후보를 탐색합니다...")
retrieved_docs = active_retriever.invoke(query) 

if not retrieved_docs:
    print("❌ 관련된 문서를 하나도 찾지 못했습니다.")
else:
    print(f"🎯 최종 정제된 후보 페이지 묶음: {len(retrieved_docs)}개 발견!\n")
    final_answer = ""
    
    # ==========================================
    # 🚀 2단계: Path B 에이전트 시각적 검증 루프 시작
    # ==========================================
    for i, doc in enumerate(retrieved_docs):
        target_pdf = doc.metadata.get('source')
        start_page = doc.metadata.get('page', 0)
        
        print("-" * 50)
        print(f"▶️ [후보 {i+1}/{len(retrieved_docs)}] {os.path.basename(target_pdf)}의 {start_page + 1}p부터 탐색")
        
        # 💡 [핵심 수정]: 순서에 맞게 파라미터를 넣고 all_docs를 확실히 전달합니다!
        answer = run_corrective_agent(query, target_pdf, start_page, all_docs, max_expansions=5)
        
        # 에이전트의 판단 결과에 따른 분기 처리
        if "WRONG_DOCUMENT" in answer:
            print("⏭️ 결과: 기업명 불일치 또는 잘못된 문서. 다음 후보로 점프!")
            continue
            
        elif "NOT_FOUND_IN_THIS_CANDIDATE" in answer:
            print("⏭️ 결과: 해당 위치(주석 또는 무관한 표)에 정답 없음. 다음 후보로 점프!")
            continue
            
        else:
            print("🎉 결과: 완벽한 정답 발견!")
            final_answer = answer
            break # 정답을 찾았으므로 루프 즉시 종료!

    # 3단계: 루프를 다 돌았는데도 못 찾았을 경우의 예외 처리
    if not final_answer:
        final_answer = "모든 문서와 후보 페이지를 정밀 조사했으나 정답 수치를 찾지 못했습니다."

    # 최종 결과 출력
    print("\n" + "="*50)
    print("🏆 최종 답변:")
    print(f" {final_answer}")
    print("="*50)


🧐 SOTA 하이브리드 에이전트 테스트 (Self-Query + Parent + BM25)
질문: LG에너지솔루션의 연결 제무재표 기준 2025년도의 당기순이익은 얼마인가요?

🤔 [Self-Query Router] 폴더에서 자동 인식한 10개 기업 중 필터 조건을 탐색합니다...
🎯 자동 필터링 조건 생성 완료: {'company': 'LG에너지솔루션'}
🛠️ 'LG에너지솔루션' 전용 하이브리드 검색기로 실시간 재조립합니다...

🔍 [Path A] 사전 필터링이 적용된 상태로 후보를 탐색합니다...
🎯 최종 정제된 후보 페이지 묶음: 10개 발견!

--------------------------------------------------
▶️ [후보 1/10] [LG에너지솔루션]사업보고서(2026.03.12).pdf의 46p부터 탐색

🔄 [Agent Loop 1] 분석 중인 페이지: [46, 47, 48]
🤔 Gemini가 질문의 의도를 분석하며 검증 중...
⏭️ 결과: 해당 위치 정답 없음 (Skip)
⏭️ 결과: 해당 위치(주석 또는 무관한 표)에 정답 없음. 다음 후보로 점프!
--------------------------------------------------
▶️ [후보 2/10] [LG에너지솔루션]사업보고서(2026.03.12).pdf의 1p부터 탐색

🔄 [Agent Loop 1] 분석 중인 페이지: [1, 2, 3]
🤔 Gemini가 질문의 의도를 분석하며 검증 중...
⏭️ 결과: 해당 위치 정답 없음 (Skip)
⏭️ 결과: 해당 위치(주석 또는 무관한 표)에 정답 없음. 다음 후보로 점프!
--------------------------------------------------
▶️ [후보 3/10] [LG에너지솔루션]사업보고서(2026.03.12).pdf의 302p부터 탐색

🔄 [Agent Loop 1] 분석 중인 페이지: [302, 303, 304]
🤔 Gemini가 질문의 의도를 분석하며 검증 중...
⏭️ 결과: 